# ⚖️ Load Balancers — Complete Guide

> w Definitier:**
> *"A Load Balancer is a traffic cop that sits in front of your servers — it receives all incoming requests and distributes them across multiple backend servers so no single server gets overwhelmed. It makes many servers look like one to the outside world."*

---

## Table of Contents

1. [Why Load Balancers Are Needed](#1-why-load-balancers-are-needed)
2. [Round Robin](#2-round-robin)
3. [Least Connections](#3-least-connections)
4. [Weighted Round Robin](#4-weighted-round-robin)
5. [IP Hash](#5-ip-hash)
6. [GEO-Based Routing](#6-geo-based-routing)
7. [Health Checks](#7-health-checks)
8. [Layer 4 vs Layer 7 Load Balancing](#8-layer-4-vs-layer-7-load-balancing)
9. [Quick Revision Cheatsheet](#9-quick-revision-cheatsheet)

---

## 1. Why Load Balancers Are Needed

### 🧠 Simple Definition
A Load Balancer is a **reverse proxy** that distributes incoming network traffic across a pool of backend servers. It is the single entry point for all client requests — clients never talk to individual servers directly.

> **Interview Line to Remember:**
> *"A load balancer solves two problems: it prevents any one server from being overwhelmed (scalability), and it routes around failed servers automatically (availability). Without it, you either have a bottleneck or a single point of failure — or both."*

---

### ❓ What Problem Does It Solve?

**Problem 1 — Single Server Bottleneck:**
One server can handle maybe 1,000 concurrent requests. Your app goes viral and 100,000 users hit it simultaneously. One server dies. Load balancer spreads those 100,000 requests across 100 servers — each handling 1,000.

**Problem 2 — Single Point of Failure:**
If your one server crashes at 2 AM, your entire app is down until someone wakes up and fixes it. With a load balancer and multiple servers, one server dying is invisible to users — traffic is instantly rerouted to healthy servers.

**Problem 3 — Deployment Downtime:**
To deploy a new version, you'd take the server down. With a load balancer, you deploy to servers one by one (rolling deployment) while the LB keeps routing traffic to the others — zero downtime.

### 🏭 Why Needed in Production?
- **High Availability (HA)** — no single point of failure; one server down = users don't notice
- **Horizontal Scaling** — add servers behind the LB to handle more traffic; no app changes needed
- **Zero-Downtime Deployments** — rolling updates, blue-green deployments, canary releases
- **SSL Termination** — LB handles HTTPS decryption so backend servers don't have to
- **DDoS Protection** — LB absorbs and filters attack traffic before it reaches servers
- **Session Management** — sticky sessions keep users on the same server if needed
- **Observability** — central point to log all traffic, measure latency, detect anomalies

### 💥 What Happens Without It?
- One server → one failure point → 100% downtime when it crashes
- Traffic surge → single server maxes out CPU → requests queue → timeouts → users see 503
- Deployment = maintenance window → you email users "site down at 2 AM" → bad experience
- No central traffic insight → impossible to rate-limit, monitor, or filter requests globally

### 🏢 Real Company Example
**Google** — when you search, your request hits one of Google's load balancers (Maglev) which routes to one of thousands of backend servers. Google's load balancer processes **millions of requests per second**. Without it, google.com would be a single server that crashes under the first 10,000 users.

---

### 📐 Load Balancer Architecture

```
Without Load Balancer:              With Load Balancer:
────────────────────────            ──────────────────────────────────
                                                    ┌─► Server 1 (healthy)
User 1 ──►                          Users ──► [LB] ─┼─► Server 2 (healthy)
User 2 ──► [Single Server] 💥       (1 IP)          └─► Server 3 (healthy)
User 3 ──►   crashes under                          (Server 4 is down — LB ignores it)
             load → all down
```

---

### 📐 What Load Balancers Also Do (Bonus Interview Points)

| Feature                  | What It Does                                                       |
|--------------------------|--------------------------------------------------------------------|
| **SSL Termination**      | Decrypts HTTPS at LB level; backend talks plain HTTP — faster      |
| **Sticky Sessions**      | Routes same user to same server (for stateful apps)               |
| **Health Checks**        | Removes unhealthy servers from rotation automatically              |
| **Rate Limiting**        | Limits requests per IP to prevent abuse/DDoS                      |
| **Compression**          | Compresses responses (gzip) before sending to client              |
| **Caching**              | Some LBs cache static responses (CDN-like behavior)               |
| **A/B Testing**          | Route % of traffic to new server version (canary deployment)      |

---

## 2. Round Robin

### 🧠 Simple Definition
Round Robin is the **simplest load balancing algorithm**. Requests are distributed to servers one by one **in a fixed circular order** — Server 1, Server 2, Server 3, Server 1, Server 2, Server 3... and repeat. Every server gets an equal turn, no matter what.

> **Interview Line to Remember:**
> *"Round Robin is dealing cards at a poker table — every server gets a card in turn, cycling through equally. Simple, stateless, zero overhead. Best when all servers are identical and all requests take similar time."*

---

### ❓ What Problem Does It Solve?
The simplest form of traffic distribution — ensures every server gets an **equal share of requests** with no complex logic. Easy to implement, easy to reason about.

### 🏭 Why Needed in Production?
- Default algorithm in Nginx, HAProxy, AWS ALB — works well for homogeneous server fleets
- Stateless — LB doesn't need to track any server state → lowest overhead
- Predictable distribution — easy to capacity plan (10 servers = each handles 10% of traffic)
- Works perfectly for stateless APIs where each request is independent and similar in cost

### 💥 What Happens Without a Distribution Strategy?
All traffic hits Server 1 until it dies → Server 2 gets everything → cascading failure.
Round Robin at least gives each server a fair turn, preventing any one server from being the permanent target.

### 💥 Round Robin's Own Weakness:
Ignores server load — Server 1 might be processing a 30-second video transcode while Server 2 is idle. Round Robin still sends the next request to Server 1, making it worse while Server 2 sits empty.

### 🏢 Real Company Example
**Nginx default configuration** — out of the box, Nginx uses Round Robin. Most startups with uniform server fleets (all same-spec EC2 instances, all handling similar REST API calls) use Round Robin without ever needing a more complex algorithm.

---

### 📐 Round Robin Visualization

```
Servers: [S1] [S2] [S3]
                                    Requests Routed:
Request 1  ──────────────────────►  S1
Request 2  ──────────────────────►  S2
Request 3  ──────────────────────►  S3
Request 4  ──────────────────────►  S1  (back to start)
Request 5  ──────────────────────►  S2
Request 6  ──────────────────────►  S3
Request 7  ──────────────────────►  S1  (cycles again)

Result: Each server handles exactly 33.3% of traffic.
Problem: Ignores that S1 might be busy processing a slow request.
```

---

## 3. Least Connections

### 🧠 Simple Definition
Least Connections routes each new request to the server with the **fewest currently active connections**. The LB tracks live connection counts per server and always picks the least-busy one.

> **Interview Line to Remember:**
> *"Least Connections is the smart waiter — it doesn't take turns blindly, it looks at which table has the fewest customers and seats the next guest there. The server with the lightest current load gets the next request."*

---

### ❓ What Problem Does It Solve?
Round Robin's blind spot: it ignores actual server load. If Server 1 is handling 10 long-running connections and Server 2 is idle, Round Robin still sends the next request to Server 1. Least Connections fixes this by **measuring actual current load**.

### 🏭 Why Needed in Production?
- **Variable request durations** — file uploads, video processing, ML inference, DB queries take wildly different times
- Prevents hot-spotting — one server processing slow requests won't keep getting new ones
- Self-balancing — as servers finish work, their connection count drops and they naturally receive more traffic
- Ideal for WebSockets and long-lived connections where "turns" don't make sense

### 💥 What Happens Without It?
With Round Robin on variable-duration requests:
- Server 1 gets 5 video transcode jobs (each takes 30 seconds) → 150 seconds of backlog
- Server 2 gets 5 quick API calls (each takes 5ms) → idle in milliseconds
- Next request goes to Server 1 (Round Robin's turn) → already overloaded → timeout

### 🏢 Real Company Example
**Zoom video calls** — each connection is a long-lived WebSocket. Some connections are low-quality (low load), others are 1080p multi-participant calls (high load). Zoom's LB uses Least Connections (or a variant) to route new calls to the least-loaded media server — not just the next in rotation.

---

### 📐 Least Connections Visualization

```
Active connections at moment of new request:

Server 1: ████████ 8 connections  (busy)
Server 2: ██ 2 connections        (lightest → gets the request ✅)
Server 3: █████ 5 connections     (moderate)

New Request ──────────────────────────────────► Server 2

After routing:
Server 1: ████████ 8
Server 2: ███ 3     (now has the new connection)
Server 3: █████ 5

Next request — Server 2 still lightest (or Server 3 if now 3=3):
LB continuously re-evaluates before each routing decision.
```

---

## 4. Weighted Round Robin

### 🧠 Simple Definition
Weighted Round Robin assigns a **weight (priority number)** to each server based on its capacity. Servers with higher weights receive proportionally more requests. A server with weight 3 gets 3 requests for every 1 request sent to a server with weight 1.

> **Interview Line to Remember:**
> *"Weighted Round Robin is Round Robin with fairness for unequal servers — stronger servers get more requests. A 16-core server should handle more traffic than a 2-core server. Weight reflects capacity; distribution reflects weight."*

---

### ❓ What Problem Does It Solve?
Standard Round Robin assumes all servers are equal. In reality, you might have:
- Old servers (2 cores, 4GB RAM) alongside new ones (16 cores, 64GB RAM)
- Servers in different regions with different specs
- Temporarily degraded servers you want to send less traffic

Weighted Round Robin lets you **match traffic to capacity**.

### 🏭 Why Needed in Production?
- **Mixed server fleets** — gradual hardware upgrades mean old and new servers coexist
- **Canary deployments** — send 5% of traffic to new version (weight=1) vs 95% to stable (weight=19)
- **Reserved capacity** — keep one server at low weight as a "warm spare" during traffic spikes
- **Cost optimization** — spot instances (cheaper, less reliable) get lower weight; on-demand get higher

### 💥 What Happens Without It?
Treating a 2-core server the same as a 16-core server with Round Robin → the 2-core server gets the same number of requests → it's overwhelmed → latency spikes → users on that server see slow responses.

### 🏢 Real Company Example
**Netflix during cloud migration** — when moving from old data centers to AWS, Netflix had old bare-metal servers (lower capacity) and new AWS instances (higher capacity) running simultaneously. Weighted Round Robin sent more traffic to the powerful new instances and less to the old ones — smooth transition with no downtime.

---

### 📐 Weighted Round Robin Visualization

```
Server Weights:
  Server 1: Weight 3  (powerful: 16 cores, 64GB RAM)
  Server 2: Weight 1  (weak:     2 cores,  4GB RAM)
  Server 3: Weight 2  (medium:   8 cores, 32GB RAM)

Total weight = 6
Distribution per 6 requests:
  Server 1 gets 3 requests  (3/6 = 50%)
  Server 2 gets 1 request   (1/6 = 17%)
  Server 3 gets 2 requests  (2/6 = 33%)

Traffic pattern:
  Req 1 → S1
  Req 2 → S1
  Req 3 → S1
  Req 4 → S2
  Req 5 → S3
  Req 6 → S3
  Req 7 → S1 (cycle repeats)

Canary Deployment Example:
  Stable v1 (99 servers): Weight 99
  New    v2 (1 server):   Weight 1
  → 1% of users get v2. If it works, gradually increase weight.
```

---

## 5. IP Hash

### 🧠 Simple Definition
IP Hash takes the **client's IP address**, runs it through a hash function, and uses the result to **always route that IP to the same server**. Same client IP = same server, every time. This is called **sticky sessions** or **session persistence**.

> **Interview Line to Remember:**
> *"IP Hash is sticky routing — it hashes the client's IP to always land on the same server. The same user always hits the same server. Essential when session data is stored locally on servers rather than in a shared cache."*

---

### ❓ What Problem Does It Solve?
Some applications store **session state locally on the server** — shopping cart in server memory, uploaded file being processed, WebSocket connection. If a user's second request goes to a different server, their state is lost.

IP Hash ensures a user's entire session sticks to one server — solving **session affinity**.

### 🏭 Why Needed in Production?
- **Stateful applications** — session data stored in server memory (not Redis)
- **File upload processing** — multi-part upload where each chunk must reach the same server
- **WebSocket connections** — subsequent HTTP requests from the same client must hit the server holding the WS connection
- **Consistent user experience** — user always gets the same regional server (lower latency through TCP reuse)

### 💥 What Happens Without It?
User adds item to cart → request goes to Server 1 → cart stored in Server 1's memory.
User clicks "Checkout" → request goes to Server 2 (Round Robin) → cart not found → empty cart → user confusion → lost sale.

### 💥 IP Hash's Weakness:
- Uneven distribution — if 80% of traffic comes from one corporate IP (office NAT), all those users hit one server
- Server failures break stickiness — if Server 1 dies, all its sticky users get redistributed

### 🏢 Real Company Example
**Online multiplayer game servers** — when you join a game lobby, your IP is hashed to a specific game server. All your subsequent packets (position updates, chat, game state) go to that same server throughout the game. Switching servers mid-game would desync your game state.

---

### 📐 IP Hash Visualization

```
Hash Function: hash(client_ip) % number_of_servers

Client A: IP 192.168.1.10 → hash = 7 → 7 % 3 = 1 → always Server 1
Client B: IP 10.0.0.5     → hash = 5 → 5 % 3 = 2 → always Server 2
Client C: IP 172.16.0.1   → hash = 9 → 9 % 3 = 0 → always Server 3

Client A sends 100 requests over the session:
  All 100 → Server 1 (never changes as long as server count stays same)

Server pool changes (Server 2 added):
  hash % 4 now → Client A might remap to different server
  → Sessions broken for some users (consistent hashing solves this)
```

> **Interview Bonus:** **Consistent Hashing** is an upgrade to IP Hash that minimizes remapping when servers are added/removed. Used in Cassandra, DynamoDB, and CDNs. Only 1/N sessions are disrupted when a server is added (instead of all of them).

---

## 6. GEO-Based Routing

### 🧠 Simple Definition
GEO-Based Routing (Geographic Routing) directs users to the **nearest server cluster based on their geographic location** — country, region, or city. A user in Mumbai hits servers in Mumbai. A user in New York hits servers in Virginia. Physics wins — closer = faster.

> **Interview Line to Remember:**
> *"GEO Routing is physics-aware load balancing — it routes users to the closest data center because speed of light is real. A Mumbai user hitting a New York server has 200ms of latency just from the distance. GEO routing eliminates that."*

---

### ❓ What Problem Does It Solve?
The speed of light is the ultimate bottleneck. Even on fiber, the round-trip time from Mumbai to New York is ~200ms — just from distance. For a typical web page requiring 10 round trips, that's 2 seconds of pure latency before any processing happens. GEO routing keeps users and servers on the same continent.

### 🏭 Why Needed in Production?
- **Latency reduction** — users routed to nearest DC get responses 5–10x faster
- **Data residency compliance** — GDPR requires EU user data to stay in EU; GEO routing enforces this
- **Load distribution** — Asia traffic hits Asia servers; Europe traffic hits Europe servers; no cross-continent bottleneck
- **Disaster recovery** — if US-East goes down, route US traffic to US-West automatically
- **Content localization** — serve country-specific content (language, currency, legal disclaimers) from regional servers

### 💥 What Happens Without It?
A company with servers only in US-East:
- Indian users see 200–300ms baseline latency → Google's threshold for "feels slow"
- GDPR violation if EU user data flows through US servers
- All global traffic hammers one region → US-East becomes bottleneck for worldwide users
- Any US-East outage → global outage

### 🏢 Real Company Example
**Netflix** — has servers (Open Connect Appliances) deployed inside ISPs in 1,000+ cities worldwide. When you press play in Bangalore, the video streams from a Netflix server *inside* your ISP's network — sometimes just a few kilometers away. This is why Netflix streams start in < 1 second. Without GEO routing, every Indian user would stream from the US — buffering hell.

---

### 📐 GEO-Based Routing Flow

```
Global DNS + GEO Routing:

User in Mumbai  ──► DNS query → GEO lookup → "India" → Route to Mumbai DC
User in London  ──► DNS query → GEO lookup → "UK"    → Route to London DC
User in New York──► DNS query → GEO lookup → "US"    → Route to Virginia DC
User in Tokyo   ──► DNS query → GEO lookup → "Japan" → Route to Tokyo DC

Latency comparison (Mumbai user):
  Without GEO: Mumbai → New York DC → Back = 200ms baseline
  With GEO:    Mumbai → Mumbai DC   → Back = 5ms baseline
  Improvement: 40x faster ⚡

Failover with GEO:
  Mumbai DC goes down → 
  GEO routing detects failure →
  Indian users rerouted to Singapore DC (next nearest) →
  Latency increases from 5ms to 40ms but app stays UP ✅
```

---

### 📐 GEO Routing vs CDN — What's the Difference?

| Feature           | GEO Load Balancing              | CDN (Content Delivery Network)         |
|------------------|----------------------------------|----------------------------------------|
| **What's routed**| Dynamic app requests             | Static assets (images, JS, CSS, video) |
| **Examples**     | AWS Route 53, Cloudflare, GCP LB | Cloudflare, Akamai, AWS CloudFront     |
| **Caches?**      | No — passes to origin server     | Yes — serves from edge cache           |
| **Best for**     | APIs, databases, dynamic content | Images, videos, scripts, HTML          |

> Both use geography. CDN caches content at the edge. GEO LB routes to the nearest full application server.

---

## 7. Health Checks

### 🧠 Simple Definition
A Health Check is the load balancer **continuously pinging each backend server** to verify it's alive and responding correctly. If a server fails the health check, the LB automatically removes it from the rotation and stops sending traffic to it until it recovers.

> **Interview Line to Remember:**
> *"Health checks are the load balancer's heartbeat monitor — it pings every server every few seconds. No response or wrong response = server marked unhealthy = traffic rerouted to healthy servers. It's the mechanism that makes high availability actually work."*

---

### ❓ What Problem Does It Solve?
Servers crash, run out of memory, get deadlocked, or have their application process die while the OS is still running. Without health checks, the LB keeps sending traffic to dead servers — users get connection refused or timeouts. Health checks make failure **self-healing and automatic**.

### 🏭 Why Needed in Production?
- **Automatic failure detection** — no human needed to notice and manually remove a dead server
- **Graceful removal** — server starting up? Don't send it traffic yet. Server shutting down? Stop new traffic first
- **Deep health checks** — check not just "is server responding" but "can server reach the database" → catches partial failures
- **Zero-downtime deployments** — deploy to one server, it fails health check during startup → LB doesn't send it traffic until it's ready
- **SLA compliance** — 99.99% uptime means < 52 minutes downtime/year; health checks + auto-rerouting makes this achievable

### 💥 What Happens Without It?
- Server crashes at 3 AM → LB has no idea → keeps sending traffic to dead server → 33% of users (on a 3-server fleet) get errors → support tickets flood in → on-call engineer woken up
- New deployment introduces a bug → application crashes → LB with no health checks sends traffic to the crashed server → users see 502s for however long it takes someone to notice

### 🏢 Real Company Example
**AWS ALB (Application Load Balancer)** — health checks are mandatory. Every 30 seconds (configurable), ALB pings `GET /health` on each EC2 instance. If 2 consecutive checks fail, the instance is removed from the target group. When it recovers and passes 3 consecutive checks, it's re-added. This happens **automatically, 24/7, without human intervention** — the foundation of AWS's 99.99% SLA.

---

### 📐 Health Check Flow

```
Normal Operation (all healthy):
  LB ──► ping S1 /health → 200 OK ✅ → keep in rotation
  LB ──► ping S2 /health → 200 OK ✅ → keep in rotation
  LB ──► ping S3 /health → 200 OK ✅ → keep in rotation

Server 2 crashes:
  LB ──► ping S2 /health → timeout ❌ (check 1 fail)
  LB ──► ping S2 /health → timeout ❌ (check 2 fail) → MARK UNHEALTHY
  LB ──► Traffic split: S1 50%, S2 0% (removed), S3 50%
  (Users on S2's connections get retried on S1 or S3)

Server 2 recovers:
  LB ──► ping S2 /health → 200 OK ✅ (check 1 pass)
  LB ──► ping S2 /health → 200 OK ✅ (check 2 pass)
  LB ──► ping S2 /health → 200 OK ✅ (check 3 pass) → MARK HEALTHY
  LB ──► Traffic split: S1 33%, S2 33%, S3 33% (S2 re-added)
```

---

### 📐 Types of Health Checks

| Type                 | What It Checks                                       | Depth    | Example                                          |
|---------------------|------------------------------------------------------|----------|--------------------------------------------------|
| **TCP Check**        | Can LB open a TCP connection to server port?         | Shallow  | Port 80 open = healthy                           |
| **HTTP Check**       | Does `GET /health` return HTTP 200?                  | Medium   | App process alive, returning responses           |
| **HTTPS Check**      | Same as HTTP but over TLS                            | Medium   | Validates SSL cert too                           |
| **Deep/Custom Check**| `/health` checks DB connection, cache, disk space   | Deep     | Returns 200 only if DB + Redis are reachable     |

---

### 📐 Health Check Endpoint Best Practice

```python
# What a production /health endpoint should check:
GET /health

Response 200 OK:
{
  "status": "healthy",
  "checks": {
    "database":   "connected",    ← can we reach PostgreSQL?
    "cache":      "connected",    ← can we reach Redis?
    "disk_space": "ok",           ← >10% free
    "memory":     "ok"            ← <90% used
  },
  "version": "v2.4.1",
  "uptime":  "3d 14h 22m"
}

Response 503 Service Unavailable (when unhealthy):
{
  "status": "unhealthy",
  "checks": {
    "database":   "failed",       ← LB sees 503 → removes server ❌
    "cache":      "connected",
    "disk_space": "ok",
    "memory":     "ok"
  }
}
```

**Health Check Configuration (AWS ALB example):**
```
Healthy threshold:   3 consecutive successes → mark healthy
Unhealthy threshold: 2 consecutive failures  → mark unhealthy
Interval:            30 seconds between checks
Timeout:             5 seconds to respond
Path:                GET /health
Success codes:       200
```

---

## 8. Layer 4 vs Layer 7 Load Balancing

> **Interview Bonus Section — frequently asked alongside LB algorithms**

### 🧠 Simple Definitions

- **Layer 4 (Transport Layer)** — Routes based on **IP address and TCP/UDP port** only. Fast but dumb — doesn't look inside the packet. Just moves bytes.
- **Layer 7 (Application Layer)** — Routes based on **HTTP content** — URL path, headers, cookies, hostname. Smart — reads the request and makes intelligent routing decisions.

> **Interview Line to Remember:**
> *"Layer 4 LB reads the envelope (IP + port) and routes it. Layer 7 LB opens the envelope, reads the letter (HTTP headers, URL, cookies), and routes based on the content. Layer 7 is smarter and more flexible; Layer 4 is faster and simpler."*

---

### 📐 L4 vs L7 Comparison

| Feature                  | Layer 4 (Transport)              | Layer 7 (Application)                    |
|--------------------------|----------------------------------|------------------------------------------|
| **Routes Based On**      | IP address, TCP/UDP port         | URL path, HTTP headers, cookies, hostname|
| **Sees HTTP content**    | ❌ No                            | ✅ Yes                                   |
| **Speed**                | Very fast (minimal processing)   | Slightly slower (inspects packets)       |
| **SSL Termination**      | ❌ Pass-through only             | ✅ Yes — terminates HTTPS                |
| **Cookie-based stickiness** | ❌ No                         | ✅ Yes                                   |
| **URL-based routing**    | ❌ No                            | ✅ Yes                                   |
| **Examples**             | AWS NLB, HAProxy TCP mode        | AWS ALB, Nginx, Cloudflare               |
| **Use Case**             | TCP apps, gaming, raw throughput | HTTP APIs, microservices, web apps       |

---

### 📐 L7 Routing Power — URL-Based Routing

```
Layer 7 LB can route based on URL path:

api.myapp.com/users/*     ──► User Service servers
api.myapp.com/products/*  ──► Product Service servers
api.myapp.com/payments/*  ──► Payment Service servers (PCI-compliant servers)
api.myapp.com/uploads/*   ──► File Upload servers (high disk I/O)

One domain, one load balancer, routes to different microservices
based on URL path — impossible with Layer 4.
```

---

## 9. Quick Revision Cheatsheet

```
LOAD BALANCER FUNDAMENTALS
────────────────────────────────────────────────────────────────────
Load Balancer  = Traffic cop distributing requests across servers.
                 Makes many servers look like one to clients.
Why needed     = Scalability (share load) + Availability (route around failures)
Without LB     = Single point of failure + bottleneck under traffic spike

ALGORITHMS
────────────────────────────────────────────────────────────────────
Round Robin          = Requests cycle through servers in order. Simple.
                       Best: identical servers, similar request durations.
                       Weakness: ignores actual server load.

Least Connections    = Route to server with fewest active connections.
                       Best: variable-length requests (uploads, streaming).
                       Weakness: needs to track state (connection counts).

Weighted Round Robin = Round Robin but servers have weights (capacity scores).
                       Best: mixed hardware fleets, canary deployments.
                       Weakness: static weights don't adapt to runtime load.

IP Hash              = Hash client IP → always same server. Sticky sessions.
                       Best: stateful apps, WebSockets, multi-part uploads.
                       Weakness: uneven if many users share one IP (NAT/office).

GEO-Based Routing    = Route user to nearest data center by geography.
                       Best: global apps, latency reduction, GDPR compliance.
                       Physics: closer server = lower latency (speed of light).

Health Checks        = LB pings servers periodically. Fail = removed from pool.
                       Types: TCP (port open?), HTTP (/health 200?), Deep (DB+cache?)
                       Threshold: 2 fails = unhealthy, 3 passes = healthy again.

LAYER 4 vs LAYER 7
────────────────────────────────────────────────────────────────────
L4 = Routes on IP + port only. Fast. Dumb. (AWS NLB)
L7 = Routes on URL, headers, cookies. Smart. Flexible. (AWS ALB, Nginx)
L7 can do: URL-based routing, SSL termination, cookie stickiness, A/B testing.

ALGORITHM DECISION GUIDE
────────────────────────────────────────────────────────────────────
All servers same spec + simple REST API?          → Round Robin
Variable request durations (uploads/streaming)?  → Least Connections
Mixed server specs or canary deploy?              → Weighted Round Robin
Stateful app / session on server?                 → IP Hash
Global users across continents?                   → GEO-Based Routing
All production systems?                           → Health Checks (always)

REAL COMPANY EXAMPLES
────────────────────────────────────────────────────────────────────
Google     → Maglev LB, millions of req/sec
AWS ALB    → L7, URL routing, mandatory health checks
Netflix    → GEO routing to ISP-embedded servers worldwide
Zoom       → Least Connections for long-lived WebRTC sessions
Nginx      → Round Robin by default, all algorithms configurable
```

---

*Made for interview prep — understand the concept, remember the line, nail the answer.*